# Session 3 — Deep Learning for Recommendations (Neural Collaborative Filtering)

## 🎯 Session Objectives

1. **Understand Neural Collaborative Filtering (NCF)**: Learn how deep learning captures complex user-item interactions
2. **Master Embeddings**: Understand initialization, joint training, and dimensionality impact
3. **Implement NCF from scratch**: Build model using PyTorch with embeddings + MLPs
4. **Feature engineering**: Extend NCF with side information (genres, user demographics, etc.)
5. **Comprehensive evaluation**: Compare NCF vs. Session 2 models (MF, SVD)

---

## 🏗️ PART A: CONCEPTUAL FOUNDATION

### A.1: Why Deep Learning? The Limitations of Matrix Factorization

**Matrix Factorization (Session 2):**
$$\hat{r}_{ui} = p_u^T \cdot q_i$$

- Assumes **linear interactions** between user and item factors
- Limited expressiveness: Can only capture dot-product relationships
- Real-world interactions are often **non-linear**

**Examples of Non-linear Patterns:**
- User preference changes based on context (e.g., "I like action movies on weekends but documentaries on weekdays")
- Item interactions matter (e.g., watching Avengers 1 makes Avengers 2 more interesting)
- Cold-start: New items have sparse interactions

**Solution: Neural Collaborative Filtering (NCF)**
- Replace linear dot product with **multi-layer neural networks**
- Learn **non-linear transformations** of user-item pairs
- Maintain the **embedding paradigm** (sparse IDs → dense vectors)

---

### A.2: What is Neural Collaborative Filtering (NCF)?

**High-Level Architecture:**

```
┌─────────────────────────────────────────────────────────────┐
│                   NCF Model Architecture                     │
├─────────────────────────────────────────────────────────────┤
│                                                               │
│  User ID → [Embedding Layer] → User Embedding (d-dim)       │
│                                ↓                              │
│                           [Concatenate]                       │
│                                ↓                              │
│  Item ID → [Embedding Layer] → Item Embedding (d-dim)       │
│                                ↓                              │
│                      [MLP Layer 1: d×2 → d]                  │
│                          ReLU + Dropout                       │
│                                ↓                              │
│                      [MLP Layer 2: d → d/2]                  │
│                          ReLU + Dropout                       │
│                                ↓                              │
│                   [Output Layer: d/2 → 1]                    │
│                       Sigmoid (0-1 range)                     │
│                                ↓                              │
│                    Prediction: r̂_ui ∈ [0, 1]                │
│                                                               │
└─────────────────────────────────────────────────────────────┘
```

**Key Concepts:**

| Concept | Definition | Purpose |
|---------|-----------|---------|
| **Embedding** | Dense vector (d dimensions) representing a sparse ID | Reduces dimensionality, captures latent factors |
| **MLP** | Multi-layer perceptron (fully connected layers) | Learns non-linear interactions between embeddings |
| **Joint Training** | All parameters (embeddings + weights) updated together | Embeddings optimize for prediction task |
| **Sigmoid Output** | Maps predictions to [0, 1] for binary classification | Treats rating prediction as likelihood |

---

### A.3: Key Insights About Embeddings & Training

#### 🔴 **FAQ 1: Are embeddings and neural network weights trained together or separately?**

**Answer: TOGETHER (Joint Training)**

```
During each training iteration:
1. Forward pass: User/Item IDs → Embeddings → MLP → Prediction
2. Compute loss (e.g., Binary Cross-Entropy)
3. Backward pass: Calculate gradients for ALL parameters:
   - Embedding weights (user & item embeddings)
   - MLP layer weights (W1, b1, W2, b2, ...)
4. Update all parameters using optimizer (Adam/SGD)
```

**Why together?** 
- If trained separately, embeddings wouldn't optimize for the prediction task
- Joint training allows embeddings to learn patterns specific to your recommendation problem
- More expressive and efficient than fixed embeddings

#### 🔴 **FAQ 2: Is there an initial embedding set? How are embeddings initialized?**

**Answer: Random Initialization (Standard Practice)**

```python
# Typical initialization in PyTorch:
user_embedding = nn.Embedding(num_users=1000, embedding_dim=32)
# → Randomly initialized from N(0, 1) distribution

item_embedding = nn.Embedding(num_items=500, embedding_dim=32)
# → Randomly initialized from N(0, 1) distribution
```

**Why random?**
- No prior knowledge exists before training
- Training will adjust embeddings based on your data
- Alternative: Pre-train embeddings from Session 2 (warm-start), but not necessary

**Initialization Details:**
- Normal distribution: `N(0, 1)` (mean=0, std=1)
- Uniform distribution: `U(-0.05, 0.05)` (optional)
- Effect: Slight impact on convergence speed, not final performance

#### 🔴 **FAQ 3: Do we only use user_id and movie_id? Can we use other attributes?**

**Answer: You CAN add other attributes (Optional Enhancement)**

**Basic NCF (User ID + Item ID only):**
```
User ID → Embedding (32-dim)    →┐
                                  ├→ MLP → Prediction
Item ID → Embedding (32-dim)    →┘
```

**Enhanced NCF (with Side Information):**
```
User ID → Embedding (32-dim)           →┐
User Age → Dense Feature (1-dim)       │
User Gender → Embedding (8-dim)        ├→ Concatenate (64-dim) → MLP → Prediction
                                        │
Item ID → Embedding (32-dim)           │
Item Genre → Embedding (16-dim)        │
Item Year → Dense Feature (1-dim)      →┘
```

**Which attributes help?**
- ✅ **User attributes**: Age, gender, location, membership type
- ✅ **Item attributes**: Genre, release year, director, runtime
- ⚠️ **Interaction attributes**: Time of day, device type, context
- ❌ **Avoid**: Highly correlated features, too many sparse categoricals

**Implementation consideration:**
- Categorical features (genre, gender) → Embeddings
- Numeric features (year, age) → Normalize & concatenate directly

---

### A.4: Training Objectives & Loss Functions

**For Binary Ratings (Like/Dislike):**
```python
Loss = Binary Cross-Entropy (BCE)
L = -y*log(ŷ) - (1-y)*log(1-ŷ)
```
- y ∈ {0, 1} (user rated item or not)
- ŷ ∈ [0, 1] (model prediction)

**For Continuous Ratings (1-5 stars):**
```python
Loss = Mean Squared Error (MSE)
L = (y - ŷ)²
```
- y ∈ [1, 5]
- Normalize predictions to [1, 5] range (no sigmoid)

---

## 🔧 PART B: IMPLEMENTATION STEPS

### B.1: Setup & Data Preparation
### B.2: Model Architecture Definition
### B.3: Training Loop
### B.4: Evaluation & Comparison
### B.5: Feature Engineering (Optional Extension)

---

## 📊 Expected Outcomes

✅ NCF model trained and evaluated
✅ Performance comparison: NCF vs MF (from Session 2)
✅ Embedding visualizations (t-SNE, PCA)
✅ Production-ready code in `src/models/neural_cf.py`


In [ ]:
## ============================================================================
## PART B.1: SETUP & IMPORTS
## ============================================================================

import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# PyTorch imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Configure device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🎯 Device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")

In [ ]:
## ============================================================================
## PART B.1 (continued): DATA LOADING & PREPARATION
## ============================================================================

# Load data from Session 1 (processed ratings)
project_root = Path().resolve().parent
DATA_PATH = project_root / 'data' / 'processed' / 'ratings.csv'

try:
    ratings_df = pd.read_csv(str(DATA_PATH))
    print(f"✅ Loaded {len(ratings_df)} ratings from {DATA_PATH}")
except FileNotFoundError:
    print(f"⚠️ File not found. Creating synthetic data for demo...")
    np.random.seed(42)
    n_users, n_items, n_ratings = 500, 200, 5000
    ratings_df = pd.DataFrame({
        'user_id': np.random.randint(0, n_users, n_ratings),
        'item_id': np.random.randint(0, n_items, n_ratings),
        'rating': np.random.choice([1, 2, 3, 4, 5], n_ratings)
    })
    # Remove duplicates, keep highest rating
    ratings_df = ratings_df.sort_values('rating', ascending=False)\
        .drop_duplicates(subset=['user_id', 'item_id'], keep='first')\
        .reset_index(drop=True)

# Display data overview
print(f"\n📊 Dataset Overview:")
print(f"  Shape: {ratings_df.shape}")
print(f"  Unique users: {ratings_df['user_id'].nunique()}")
print(f"  Unique items: {ratings_df['item_id'].nunique()}")
print(f"  Rating range: [{ratings_df['rating'].min()}, {ratings_df['rating'].max()}]")
print(f"  Sparsity: {(1 - len(ratings_df) / (ratings_df['user_id'].nunique() * ratings_df['item_id'].nunique())) * 100:.2f}%")
print(f"\n  Rating distribution:\n{ratings_df['rating'].value_counts().sort_index()}")
print(f"\n  First 5 rows:")
print(ratings_df.head())

# ============================================================================
# STEP 1: INDEX MAPPING (IDs may not be 0-indexed)
# ============================================================================
# Create mappings: Original ID → Index [0, num_items)
user_id_to_idx = {uid: idx for idx, uid in enumerate(sorted(ratings_df['user_id'].unique()))}
item_id_to_idx = {iid: idx for idx, iid in enumerate(sorted(ratings_df['item_id'].unique()))}

ratings_df['user_idx'] = ratings_df['user_id'].map(user_id_to_idx)
ratings_df['item_idx'] = ratings_df['item_id'].map(item_id_to_idx)

num_users = len(user_id_to_idx)
num_items = len(item_id_to_idx)

print(f"\n✅ Index mapping created:")
print(f"  Users: {num_users} (indices 0-{num_users-1})")
print(f"  Items: {num_items} (indices 0-{num_items-1})")

# ============================================================================
# STEP 2: NORMALIZE RATINGS TO [0, 1] FOR BINARY CROSS-ENTROPY
# ============================================================================
# Binary NCF: Treat high ratings (4-5) as positive, low (1-2) as negative
# Alternative: Normalize continuous ratings to [0, 1]

# Option A: Binary (recommended for implicit feedback)
ratings_df['rating_binary'] = (ratings_df['rating'] >= 3).astype(int)
print(f"\n✅ Binary ratings: {ratings_df['rating_binary'].value_counts().to_dict()}")

# Option B: Normalize to [0, 1]
scaler = MinMaxScaler(feature_range=(0, 1))
ratings_df['rating_normalized'] = scaler.fit_transform(ratings_df[['rating']])

# ============================================================================
# STEP 3: TRAIN/VAL/TEST SPLIT
# ============================================================================
train_df, temp_df = train_test_split(ratings_df, test_size=0.3, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"\n✅ Data splits:")
print(f"  Train: {len(train_df)} ({len(train_df)/len(ratings_df)*100:.1f}%)")
print(f"  Val:   {len(val_df)} ({len(val_df)/len(ratings_df)*100:.1f}%)")
print(f"  Test:  {len(test_df)} ({len(test_df)/len(ratings_df)*100:.1f}%)")

In [ ]:
## ============================================================================
## PART B.1 (continued): PYTORCH DATASET & DATALOADER
## ============================================================================

class RatingsDataset(Dataset):
    """PyTorch Dataset for NCF model training.
    
    Returns: (user_idx, item_idx, rating)
    where indices are 0-indexed and rating is normalized to [0,1] or binary [0,1]
    """
    def __init__(self, df: pd.DataFrame, use_binary=True):
        self.user_idx = torch.LongTensor(df['user_idx'].values)
        self.item_idx = torch.LongTensor(df['item_idx'].values)
        
        # Use binary or normalized ratings
        rating_col = 'rating_binary' if use_binary else 'rating_normalized'
        self.ratings = torch.FloatTensor(df[rating_col].values)
        
    def __len__(self):
        return len(self.user_idx)
    
    def __getitem__(self, idx):
        return self.user_idx[idx], self.item_idx[idx], self.ratings[idx]


# Create datasets
train_dataset = RatingsDataset(train_df, use_binary=True)
val_dataset = RatingsDataset(val_df, use_binary=True)
test_dataset = RatingsDataset(test_df, use_binary=True)

# Create dataloaders
BATCH_SIZE = 128

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"✅ DataLoaders created:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Test batches: {len(test_loader)}")

# Inspect a sample batch
sample_user, sample_item, sample_rating = next(iter(train_loader))
print(f"\n📦 Sample batch (first batch):")
print(f"  User indices: {sample_user[:5].tolist()}")
print(f"  Item indices: {sample_item[:5].tolist()}")
print(f"  Ratings: {sample_rating[:5].tolist()}")
print(f"  Shapes: user={sample_user.shape}, item={sample_item.shape}, rating={sample_rating.shape}")

In [ ]:
## ============================================================================
## PART B.2: MODEL ARCHITECTURE - NEURAL COLLABORATIVE FILTERING
## ============================================================================

class NeuralCF(nn.Module):
    """
    Neural Collaborative Filtering model.
    
    Architecture:
        User ID (sparse) → Embedding (d) ──┐
                                            ├→ Concatenate (2d) → MLP → Sigmoid → Rating
        Item ID (sparse) → Embedding (d) ──┘
    
    Key insight: Embeddings are jointly trained with MLP weights during backprop.
    """
    def __init__(self, num_users, num_items, embedding_dim=32, hidden_dims=[64, 32, 16], 
                 dropout_rate=0.3, use_batchnorm=False):
        """
        Args:
            num_users: Number of users (for embedding table size)
            num_items: Number of items (for embedding table size)
            embedding_dim: Dimension of embeddings (typically 32-128)
            hidden_dims: Dimensions of MLP hidden layers
            dropout_rate: Dropout probability for regularization
            use_batchnorm: Whether to use batch normalization
        """
        super(NeuralCF, self).__init__()
        
        # TODO: Initialize embedding layers
        # Create:
        # 1. self.user_embedding = nn.Embedding(num_users, embedding_dim)
        # 2. self.item_embedding = nn.Embedding(num_items, embedding_dim)
        # These are randomly initialized and will be jointly trained
        
        # TODO: Build MLP layers
        # Steps:
        # 1. Initialize empty list: mlp_layers = []
        # 2. Set input_dim = embedding_dim * 2 (concatenated embeddings)
        # 3. For each hidden_dim in hidden_dims:
        #    - Add nn.Linear(input_dim, hidden_dim)
        #    - If use_batchnorm: Add nn.BatchNorm1d(hidden_dim)
        #    - Add nn.ReLU()
        #    - Add nn.Dropout(dropout_rate)
        #    - Update input_dim = hidden_dim
        # 4. self.mlp = nn.Sequential(*mlp_layers)
        
        # TODO: Add output layer
        # 1. self.output_layer = nn.Linear(input_dim, 1)
        # 2. self.sigmoid = nn.Sigmoid()
        pass
    
    def forward(self, user_idx, item_idx):
        """Forward pass: User/Item IDs → Embeddings → MLP → Prediction
        
        During training:
        - Embedding lookup creates differentiable connection
        - MLP learns non-linear transformations
        - Sigmoid squashes output to [0, 1]
        - All parameters get gradient updates
        """
        
        # TODO: Implement forward pass
        # Steps:
        # 1. Get user embedding: user_emb = self.user_embedding(user_idx)
        # 2. Get item embedding: item_emb = self.item_embedding(item_idx)
        # 3. Concatenate: combined = torch.cat([user_emb, item_emb], dim=1)
        # 4. Pass through MLP: hidden = self.mlp(combined)
        # 5. Output layer: logits = self.output_layer(hidden)
        # 6. Apply sigmoid: predictions = self.sigmoid(logits).squeeze(1)
        # 7. Return predictions
        pass


# TODO: Create model instance
# Steps:
# 1. Set EMBEDDING_DIM = 32
# 2. Set HIDDEN_DIMS = [64, 32, 16]
# 3. Set DROPOUT_RATE = 0.3
# 4. Instantiate model: model = NeuralCF(...)
# 5. Move to device: .to(DEVICE)

# YOUR CODE HERE

# TODO: Test forward pass
# Steps:
# 1. Set model to eval mode
# 2. Get a sample batch from train_loader
# 3. Move batch to device
# 4. With torch.no_grad():
#    - Call model(user_idx, item_idx)
#    - Check output shape and range
# 5. Print results

# YOUR CODE HERE

In [ ]:
## ============================================================================
## PART B.3: TRAINING LOOP (JOINT EMBEDDING & WEIGHT TRAINING)
## ============================================================================

class NCFTrainer:
    """Trainer for Neural Collaborative Filtering model.
    
    Key principle: During each iteration, all parameters (embeddings + MLP weights)
    receive gradient updates simultaneously via backpropagation.
    """
    
    def __init__(self, model, device, learning_rate=0.001):
        self.model = model
        self.device = device
        self.criterion = nn.BCELoss()
        
        # TODO: Create optimizer
        # Use: optim.Adam(model.parameters(), lr=learning_rate)
        # This optimizer updates ALL parameters (embeddings + weights) jointly
        self.optimizer = None  # YOUR CODE HERE
        
        self.train_losses = []
        self.val_losses = []
        self.best_val_loss = float('inf')
    
    def train_epoch(self, train_loader):
        """Train for one epoch. All gradients flow through embeddings & MLP."""
        self.model.train()
        total_loss = 0.0
        
        # TODO: Training loop
        # For each batch in train_loader:
        # 1. Move batch to device: user_idx, item_idx, ratings
        # 2. Zero gradients: self.optimizer.zero_grad()
        # 3. Forward pass: predictions = self.model(user_idx, item_idx)
        # 4. Compute loss: loss = self.criterion(predictions, ratings)
        # 5. BACKWARD PASS (KEY): loss.backward()
        #    This computes gradients for embeddings AND MLP weights jointly
        # 6. Parameter update: self.optimizer.step()
        #    This updates embeddings AND MLP weights simultaneously
        # 7. Accumulate loss: total_loss += loss.item()
        
        # YOUR CODE HERE
        
        avg_loss = total_loss / len(train_loader)
        self.train_losses.append(avg_loss)
        return avg_loss
    
    def validate(self, val_loader):
        """Evaluate on validation set (no gradients)."""
        self.model.eval()
        total_loss = 0.0
        
        # TODO: Validation loop
        # With torch.no_grad():
        #   For each batch in val_loader:
        #   1. Move batch to device
        #   2. Forward pass: predictions = self.model(user_idx, item_idx)
        #   3. Compute loss: loss = self.criterion(predictions, ratings)
        #   4. Accumulate: total_loss += loss.item()
        
        # YOUR CODE HERE
        
        avg_loss = total_loss / len(val_loader)
        self.val_losses.append(avg_loss)
        return avg_loss
    
    def train_with_early_stopping(self, train_loader, val_loader, num_epochs=20, patience=3):
        """Train with early stopping to prevent overfitting."""
        print(f"\n🚀 Training NCF for {num_epochs} epochs...")
        print(f"{'Epoch':<8} {'Train Loss':<15} {'Val Loss':<15} {'Status':<20}")
        print("-" * 60)
        
        patience_counter = 0
        
        # TODO: Training loop with early stopping
        # For each epoch from 0 to num_epochs:
        # 1. Call train_epoch(train_loader) → train_loss
        # 2. Call validate(val_loader) → val_loss
        # 3. Check if val_loss < best_val_loss:
        #    - If yes: update best_val_loss, reset patience_counter, save model state
        #    - If no: increment patience_counter
        # 4. Print epoch stats
        # 5. If patience_counter >= patience:
        #    - Print early stopping message
        #    - Break out of loop
        # 6. After training, restore best model state
        
        # YOUR CODE HERE
        
        print(f"\n✅ Training complete!")


# TODO: Create trainer and train
# Steps:
# 1. trainer = NCFTrainer(model, DEVICE, learning_rate=0.001)
# 2. Set NUM_EPOCHS = 15, EARLY_STOP_PATIENCE = 3
# 3. Call trainer.train_with_early_stopping(...)

# YOUR CODE HERE

In [ ]:
## ============================================================================
## PART B.3 (continued): PLOT TRAINING CURVES
## ============================================================================

# TODO: Visualize training progress
# Create figure with training and validation loss curves
# Steps:
# 1. plt.figure(figsize=(10, 5))
# 2. plt.plot(trainer.train_losses, label='Train Loss', marker='o', markersize=4)
# 3. plt.plot(trainer.val_losses, label='Validation Loss', marker='s', markersize=4)
# 4. plt.axhline(y=trainer.best_val_loss, color='g', linestyle='--', label=f'Best Val Loss')
# 5. Add labels, title, legend, grid
# 6. plt.tight_layout() and plt.show()

# YOUR CODE HERE

print(f"✅ Training curves plotted")

In [ ]:
## ============================================================================
## PART B.4: EVALUATION ON TEST SET
## ============================================================================

def evaluate_model(model, test_loader, device):
    """Generate predictions on test set."""
    model.eval()
    all_predictions = []
    all_ratings = []
    
    with torch.no_grad():
        for user_idx, item_idx, ratings in test_loader:
            user_idx = user_idx.to(device)
            item_idx = item_idx.to(device)
            predictions = model(user_idx, item_idx)
            all_predictions.extend(predictions.cpu().numpy())
            all_ratings.extend(ratings.numpy())
    
    return np.array(all_predictions), np.array(all_ratings)


# TODO: Get test predictions and compute metrics
# Steps:
# 1. Call evaluate_model(model, test_loader, DEVICE)
#    → Get test_predictions, test_ratings
# 2. Import from sklearn.metrics: mean_squared_error, mean_absolute_error, roc_auc_score
# 3. Compute:
#    - test_mse = mean_squared_error(test_ratings, test_predictions)
#    - test_rmse = np.sqrt(test_mse)
#    - test_mae = mean_absolute_error(test_ratings, test_predictions)
#    - test_auc = roc_auc_score(test_ratings, test_predictions)
# 4. Print results

# YOUR CODE HERE

print(f"✅ Test evaluation complete")

## ============================================================================
## PART B.5: ANALYSIS & VISUALIZATION
## ============================================================================

### B.5.1: Prediction vs Actual Rating Visualization

Visualize how well the model predictions match true ratings.

In [ ]:
## B.5.1: Prediction vs Actual Rating Visualization

# TODO: Create visualization comparing predictions vs true ratings
# Steps:
# 1. Create figure with 2 subplots: fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# 
# 2. Subplot 1 (Scatter plot):
#    - axes[0].scatter(test_ratings, test_predictions, alpha=0.3, s=10)
#    - axes[0].plot([0, 1], [0, 1], 'r--', label='Perfect prediction')
#    - axes[0].set_xlabel('True Rating')
#    - axes[0].set_ylabel('Predicted Rating')
#    - axes[0].set_title('Predictions vs True Ratings')
#    - axes[0].legend() and axes[0].grid(alpha=0.3)
#
# 3. Subplot 2 (Residuals histogram):
#    - residuals = test_predictions - test_ratings
#    - axes[1].hist(residuals, bins=30, edgecolor='black')
#    - axes[1].axvline(x=0, color='r', linestyle='--', label='Zero error')
#    - axes[1].set_xlabel('Prediction Error')
#    - axes[1].set_ylabel('Frequency')
#    - axes[1].set_title('Distribution of Errors')
#    - axes[1].legend() and axes[1].grid(alpha=0.3)
#
# 4. plt.tight_layout() and plt.show()

# YOUR CODE HERE

print("✅ Prediction visualization complete")

---

## B.5.2: Ranking Metrics (Precision@K, Recall@K, NDCG@K)

**Ranking metrics** evaluate recommendation quality differently from regression metrics:
- Threshold predictions (e.g., > 0.5 = positive recommendation)
- Evaluate ranking quality for top-K items
- More aligned with real recommendation scenarios

We compare against a popularity baseline from Session 2 to understand NCF improvements.

In [ ]:
## B.5.2: Ranking Metrics

# Helper function to compute ranking metrics
def compute_ranking_metrics(predicted_ratings, true_ratings, k_values=[5, 10], threshold=0.5):
    """Compute Precision@K, Recall@K, NDCG@K for ranking evaluation.
    
    Args:
        predicted_ratings: Array of predictions
        true_ratings: Array of true labels (1 = positive, 0 = negative)
        k_values: List of K values to evaluate
        threshold: Threshold for positive prediction
    
    Returns:
        Dictionary with metrics for each K value
    """
    metrics = {}
    
    # TODO: Implement ranking metrics computation
    # For each k in k_values:
    # 1. Get top-k indices from predicted_ratings (argsort descending)
    # 2. Get top-k true labels
    # 3. Compute precision@k: sum(true[top_k]) / k
    # 4. Compute recall@k: sum(true[top_k]) / sum(true)
    # 5. Store in metrics dict
    
    # YOUR CODE HERE
    
    return metrics


# TODO: Evaluate ranking performance
# Steps:
# 1. Create test_df = test.copy()
# 2. Add predictions: test_df['prediction'] = test_predictions
# 3. Set THRESHOLD = 0.5
# 4. Create binary labels: test_df['positive'] = (test_df['rating'] > THRESHOLD).astype(int)
# 5. Call compute_ranking_metrics(...)
# 6. Print results for k=5 and k=10

# YOUR CODE HERE

print(f"✅ Ranking metrics computed")

---

## B.5.3: Embedding Visualization & Analysis

Embeddings learned during training encode latent patterns. We visualize them using PCA/t-SNE
to understand what the model learned about users and items.

**Expected patterns:**
- Items with similar genres cluster together
- Users with similar preferences cluster together
- Embeddings use multiple dimensions (not collapsed)

## B.5.3 (continued): Code Implementation

---

## B.5.4: Recommendation Examples & Model Serialization

Generate top-K recommendations for users and save the trained model for production deployment.

In [ ]:
## B.5.4: Get Recommendations for a User

# Create inverse map for item IDs
inv_item_id_map = {v: k for k, v in item_id_to_idx.items()}  # index → original ID
inv_user_id_map = {v: k for k, v in user_id_to_idx.items()}  # index → original ID


def get_recommendations(model, user_id, user_id_to_idx, item_id_to_idx, 
                       inv_item_id_map, train_df, device, k=10):
    """Generate top-K recommendations for a user.
    
    Args:
        model: Trained NCF model
        user_id: Original user ID
        user_id_to_idx: Mapping from user ID to index
        item_id_to_idx: Mapping from item ID to index
        inv_item_id_map: Mapping from item index to original ID
        train_df: Training dataframe (to exclude already-rated items)
        device: Torch device
        k: Number of recommendations
    
    Returns:
        DataFrame with recommended items and scores
    
    TODO: Implement recommendation function
    Steps:
    1. Check if user_id in user_id_to_idx, return empty DataFrame if not found
    2. Get user_idx from user_id_to_idx[user_id]
    3. Create tensors for all items:
       - user_tensor = torch.LongTensor([user_idx] * len(item_id_to_idx))
       - item_tensor = torch.LongTensor(range(len(item_id_to_idx)))
    4. Move tensors to device
    5. With torch.no_grad():
       - Get scores using model(user_tensor, item_tensor)
       - Convert to numpy
    6. Get already-rated items by this user from train_df
    7. Create DataFrame: {'item_id': [inv_item_id_map[i] for i, ...],
                          'score': scores}
    8. Filter out already-rated items
    9. Sort by score descending, return top-k
    """
    # YOUR CODE HERE
    pass


# TODO: Get recommendations for example users
# Steps:
# 1. Select a user_id from the training data: 
#    example_user_id = train_df['user_id'].iloc[0]
# 2. Call get_recommendations(model, example_user_id, ...)
# 3. Display results
# 4. Optionally: try multiple users

# YOUR CODE HERE

print(f"✅ Recommendations generated")

---

## B.5.5: Save & Load Model (Production Deployment)

In [ ]:
## B.5.5: Save and Load Model

# TODO: Save model checkpoint
# Steps:
# 1. Create models directory if not exists:
#    MODELS_PATH = project_root / 'data' / 'models'
#    MODELS_PATH.mkdir(parents=True, exist_ok=True)
# 2. Save model state dict:
#    torch.save(model.state_dict(), MODELS_PATH / 'ncf_model.pt')
# 3. Save training info (optional):
#    info = {
#        'num_users': num_users,
#        'num_items': num_items,
#        'embedding_dim': EMBEDDING_DIM,
#        'hidden_dims': HIDDEN_DIMS,
#        'best_val_loss': trainer.best_val_loss
#    }
#    with open(MODELS_PATH / 'model_info.json', 'w') as f:
#        json.dump(info, f, indent=2)

# YOUR CODE HERE

# TODO: Load model for inference
# Steps:
# 1. Create new model instance with same architecture
# 2. Load state dict: model.load_state_dict(torch.load(...))
# 3. Set to eval mode: model.eval()
# 4. Test with get_recommendations(...)

# YOUR CODE HERE

print("✅ Model saved and loaded successfully")

In [ ]:
## B.5.3 (continued): Embedding Visualization Code

from sklearn.decomposition import PCA

# TODO: Extract embeddings from trained model
# Steps:
# 1. Create method in NeuralCF class to get embeddings:
#    def get_user_embeddings(self):
#        return self.user_embedding.weight
#    def get_item_embeddings(self):
#        return self.item_embedding.weight
# 2. Extract: user_emb = model.get_user_embeddings().cpu().detach().numpy()
# 3. Extract: item_emb = model.get_item_embeddings().cpu().detach().numpy()

# YOUR CODE HERE

# TODO: Apply PCA for 2D visualization
# For users:
# - Create pca_user = PCA(n_components=2)
# - Transform: user_emb_2d = pca_user.fit_transform(user_embeddings)
# For items:
# - Create pca_item = PCA(n_components=2)
# - Transform: item_emb_2d = pca_item.fit_transform(item_embeddings)

# YOUR CODE HERE

# TODO: Plot embeddings
# Create 2 subplots showing scatter plots:
# - Subplot 1: User embeddings (PCA 2D)
# - Subplot 2: Item embeddings (PCA 2D)
# Color by index or other features if available

# YOUR CODE HERE

print("✅ Embedding visualization complete")

---

## Expected Improvements over Session 2

**NCF vs Matrix Factorization (Session 2):**
- Non-linear interactions: MLP learns complex patterns beyond dot product
- Better ranking metrics: 5-15% improvement typical for Precision@K, Recall@K
- More parameters: Trade-off for expressiveness (risk of overfitting)
- Slower inference: Extra MLP layers add latency

This section compares actual performance metrics.

In [ ]:
 # TODO: Compute ranking metrics for NCF
# This is part of the compute_ranking_metrics() function above
# Implement the full function with precision@k, recall@k, ndcg@k calculations

# Steps:
# 1. For each k in k_values:
#    a. Sort predictions in descending order, get top-k indices
#    b. Get top-k true labels
#    c. Count positives in top-k: num_positive = sum(top_k_labels)
#    d. Precision@k = num_positive / k
#    e. Recall@k = num_positive / total_positives_in_set
#    f. NDCG@k = compute_ndcg_score(top_k_labels, k)
# 2. Aggregate metrics across all test users
# 3. Return average metrics

# Then evaluate:
# - NCF ranking metrics
# - Compare against Session 2 models if available
# - Create comparison table

# YOUR CODE HERE

print(f"🎯 Ranking Metrics Comparison:")

In [ ]:
## Regression Metrics

# TODO: Compute regression metrics (RMSE, MAE) for NCF on test set
# From evaluate_model() above, you have:
# - test_predictions: Array of predictions
# - test_ratings: Array of true ratings
#
# Compute:
# 1. mse = mean_squared_error(test_ratings, test_predictions)
# 2. rmse = sqrt(mse) or use: rmse = np.sqrt(mean_squared_error(...))
# 3. mae = mean_absolute_error(test_ratings, test_predictions)
# 4. Print results with interpretation

# YOUR CODE HERE

print(f"✅ Regression metrics computed")

In [ ]:
## Regression Metrics Implementation

# TODO: Compute and display RMSE and MAE
# Steps:
# 1. Import from sklearn.metrics: mean_squared_error, mean_absolute_error
# 2. Compute metrics on test set:
#    mse = mean_squared_error(test_ratings, test_predictions)
#    rmse = np.sqrt(mse)
#    mae = mean_absolute_error(test_ratings, test_predictions)
# 3. Create comparison table with Session 2 models if available
# 4. Visualize errors (residuals histogram)

# YOUR CODE HERE

print(f"✅ Metrics computed")

In [ ]:
## Comparison with Session 2 Models

# TODO: Load Session 2 models (SVD, ALS) if available and compare
# Steps:
# 1. Load popularity baseline from Session 1
# 2. Load SVD model from Session 2 if available
# 3. Compute rankings on same test set
# 4. Create comparison dataframe:
#    - Model names (Popularity, SVD, ALS, NCF)
#    - RMSE, MAE, AUC
#    - Precision@5, @10
#    - Recall@5, @10
# 5. Visualize comparison (bar charts)
# 6. Conclusions about NCF performance

# YOUR CODE HERE

print(f"✅ Model comparison complete")

In [ ]:
## Visualization: Model Comparison

# TODO: Create comparison visualizations
# Steps:
# 1. Create metrics comparison dataframe
# 2. Plot bar charts comparing:
#    - RMSE across models
#    - Precision@K for different k values
#    - Recall@K for different k values
# 3. Create table with model performance summary

# YOUR CODE HERE

print(f"✅ Comparison visualizations created")

In [ ]:
## Extended NCF with Side Features (Optional)

# TODO: Implement Extended NCF with item features (e.g., genres)
# This is an optional advanced section
#
# Steps:
# 1. Create ExtendedNeuralCF class that:
#    - Takes genre embeddings as additional input
#    - Concatenates all embeddings before MLP
# 2. Modify Dataset to include genre information
# 3. Train extended model on same data
# 4. Compare performance with basic NCF
#
# This demonstrates that embeddings from additional features
# are also jointly trained during backpropagation

# YOUR CODE HERE

print(f"✅ Extended NCF demonstration (optional)")

---

## ============================================================================
## PART C: KEY TAKEAWAYS & COMPARISON
## ============================================================================

### ✅ Key Learning Outcomes

#### 1. **Joint Training of Embeddings & MLPs**
- **Embeddings and MLP weights trained together** during backprop
- `loss.backward()` computes gradients for all parameters simultaneously
- `optimizer.step()` updates all parameters jointly
- This is more effective than training embeddings and MLP separately

#### 2. **Embedding Initialization & Learning**
- Embeddings start **randomly** from N(0,1)
- No pre-set initial embeddings
- Training refines random vectors into meaningful representations
- Embeddings optimize for the **specific recommendation task**

#### 3. **Input Features: ID-Only vs. Extended**
- **Basic NCF**: User ID + Item ID (sufficient for pure collaborative filtering)
- **Extended NCF**: Add genres, demographics, year, etc. (better for cold-start)
- All features get embeddings, all trained jointly
- Feature engineering is important but optional

#### 4. **Why NCF > Matrix Factorization**
| Aspect | MF (Session 2) | NCF (Session 3) |
|--------|---|---|
| Interaction model | Linear: $p_u^T q_i$ | Non-linear: MLP(concat($e_u$, $e_i$)) |
| Expressiveness | Limited to dot products | Arbitrary non-linear functions |
| Performance | Baseline (good for sparse) | Better with dense interactions |
| Inference speed | Fast | Slightly slower (extra MLP layers) |
| Training | Simple, converges fast | More complex, needs regularization |

#### 5. **Evaluation Metrics**
- **Regression**: RMSE, MAE (measure prediction accuracy)
- **Ranking**: Precision@K, Recall@K, NDCG@K (measure recommendation quality)
- **Classification**: AUC (measure binary classification performance)
- Compare NCF against Session 2 baselines

### 📋 Implementation Checklist

* [ ] Part B.1: Data loaded and prepared (indexed, normalized, split)
* [ ] Part B.2: Model architecture implemented (embeddings + MLP)
* [ ] Part B.2: Forward pass tested with sample batch
* [ ] Part B.3: NCFTrainer class with joint training loop
* [ ] Part B.3: Training loop runs, loss decreases
* [ ] Part B.3: Early stopping prevents overfitting
* [ ] Part B.3: Training curves plotted
* [ ] Part B.4: Test set evaluation completed
* [ ] Part B.5.1: Prediction vs actual visualization
* [ ] Part B.5.2: Ranking metrics computed
* [ ] Part B.5.3: Embedding visualization (PCA/t-SNE)
* [ ] Part B.5.4: Top-K recommendations for sample users
* [ ] Part B.5.5: Model saved and loaded successfully
* [ ] Part C: Performance compared with Session 2 models

### 🚀 Next Steps

1. **Hyperparameter Tuning**: Optimize embedding_dim, hidden_dims, dropout, lr
2. **Cold-Start Handling**: Add item/user features for new entities
3. **Implicit Feedback**: Modify for view counts, clicks instead of explicit ratings
4. **Temporal Dynamics**: Add time-aware embeddings for concept drift
5. **Production**: Deploy model API, monitor performance, retrain periodically